> 在线讲义：[第 37 章 · 注意力的数学：√d 的方差推导、softmax 的饱和、核平滑、多头的低秩、因果 mask](https://wikieden.github.io/math-and-eval-stats/lessons/37-attention-math.html) · 本书《数学与实验统计》

# 第 37 章 · 注意力的数学：√d 的方差推导、softmax 的饱和、核平滑、多头的低秩、因果 mask

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/37-attention-math.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 方差 | √d | `dot_std(d, scaled)` | 4、8、16 → 1 |
| 第二步 饱和 | p(1 − p) | `saturation(16)` | 1.1e-7 |
| 第三步 核平滑 | NW | `kernel_smoother(1.0, tau)` | 0.8147、0.7529、0.3931 |
| 第四步 秩 | rank | `head_rank()` | 64 |
| 第五步 mask | −∞ | `causal_weights()` | 下三角，行和 1 |

例子：softmax 在不同缩放下的饱和；一维“查表”：键是位置、值是 sin，看注意力 = 核平滑；多头的秩；因果 mask。
里程碑 ME-M24 的“模拟点积方差随 d 线性增长”留给自己做；这里只用公式。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

**函数 `softmax`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def softmax(z: np.ndarray) -> np.ndarray:
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

**函数 `dot_std`**：q、k 各分量独立、均值 0、方差 1：Var(q·k) = d；除以 √d 后方差 1。返回标准差。

In [ ]:
def dot_std(d: int, scaled: bool) -> float:
    """q、k 各分量独立、均值 0、方差 1：Var(q·k) = d；除以 √d 后方差 1。返回标准差。"""
    return 1.0 if scaled else float(np.sqrt(d))

**函数 `saturation`**：logits = scale·z：返回 (最大概率, 熵（nat）, softmax 雅可比的最大元素 p_i(1 − p_i))。

In [ ]:
def saturation(scale: float, z=(2.0, 1.0, 0.0, -1.0)) -> tuple[float, float, float]:
    """logits = scale·z：返回 (最大概率, 熵（nat）, softmax 雅可比的最大元素 p_i(1 − p_i))。"""
    p = softmax(scale * np.array(z))
    return float(p.max()), float(-(p * np.log(p + 1e-300)).sum()), float((p * (1 - p)).max())

**函数 `kernel_smoother`**：键 k_j = 位置 x_j（[0, 2π] 上 21 个点），值 v_j = sin x_j。注意力 logits = −(q − k)²/(2τ²)，

In [ ]:
def kernel_smoother(q: float, tau: float, n: int = 21) -> tuple[float, float]:
    """键 k_j = 位置 x_j（[0, 2π] 上 21 个点），值 v_j = sin x_j。注意力 logits = −(q − k)²/(2τ²)，
    即 softmax(q·k/τ² − k²/(2τ²) + 常数)：就是高斯核的 Nadaraya–Watson 平滑。返回 (输出, sin q)。"""
    x = np.linspace(0, 2 * np.pi, n)
    w = softmax(-((q - x) ** 2) / (2 * tau ** 2))
    return float(w @ np.sin(x)), float(np.sin(q))

**函数 `head_rank`**：一个头的注意力打分矩阵 W_Q W_Kᵀ（d_model × d_model）的秩 ≤ d_head。

In [ ]:
def head_rank(d_model: int = 512, d_head: int = 64, seed: int = 0) -> int:
    """一个头的注意力打分矩阵 W_Q W_Kᵀ（d_model × d_model）的秩 ≤ d_head。"""
    rng = np.random.default_rng(seed)
    wq, wk = rng.standard_normal((d_model, d_head)), rng.standard_normal((d_model, d_head))
    return int(np.linalg.matrix_rank(wq @ wk.T))

**函数 `causal_weights`**：上三角（未来）位置填 −∞ 再 softmax：每行只在 j ≤ i 上有权重。

In [ ]:
def causal_weights(n: int = 4, seed: int = 0) -> np.ndarray:
    """上三角（未来）位置填 −∞ 再 softmax：每行只在 j ≤ i 上有权重。"""
    rng = np.random.default_rng(seed)
    s = rng.standard_normal((n, n))
    s[np.triu_indices(n, 1)] = -np.inf
    return softmax(s)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 为什么除以 √d：点积的方差是 d")
for d in (16, 64, 256):
    print(f"  d = {d}：q·k 的标准差 {dot_std(d, False):.0f}，除以 √d 后 {dot_std(d, True):.0f}")

In [ ]:
print("\n第二步 softmax 的温度与饱和")
for s in (0.25, 1, 4, 16):
    m, h, g = saturation(s)
    print(f"  logits = {s}×[2, 1, 0, −1]：最大概率 {m:.4f}，熵 {h:.4f} nat，最大局部梯度 p(1 − p) = {g:.2e}")

In [ ]:
print("\n第三步 注意力 = 核平滑（软查表）")
for tau in (0.1, 0.5, 2.0):
    o, t = kernel_smoother(1.0, tau)
    print(f"  查询 q = 1、核宽 τ = {tau}：输出 {o:.4f}（真值 sin 1 = {t:.4f}）")

In [ ]:
print("\n第四步 多头 = 若干个低秩打分矩阵")
print(f"  d_model = 512、d_head = 64：W_Q W_Kᵀ 是 512×512，但秩只有 {head_rank()}")
print(f"  8 个头的 Q、K、V 参数 3 × 512 × 64 × 8 = {3 * 512 * 64 * 8:,}，和一个 512 维的头一样多")

In [ ]:
print("\n第五步 因果 mask：未来位置填 −∞")
w = causal_weights()
print(f"  4×4 权重（每行和 = {w.sum(1).round(6).tolist()}）：\n{np.round(w, 3)}")
print(f"  n = 4096 时可见的 (i, j) 对：{4096 * 4097 // 2:,}，约占 n² 的一半")

## 3. 代码里才看得到的

1. **softmax 先减行最大值**：mask 后每行至少有对角线一个有限值，减最大值不会出现 −∞ − (−∞) = nan。
2. **第一步没有模拟**：Var(q·k) = d 直接由公式给出，代码只返回 √d；用随机数验证它是留给自己的练习（里程碑 ME-M24）。
3. **核平滑只有 21 个键**：τ = 0.1 时间距 0.31 比核宽大，输出基本就是最近那个键的值，所以误差没有继续变小。

## 4. 自己改一改

1. 把键加密到 201 个，重看三个 τ 的输出。
2. 把 `head_rank` 的 d_head 改成 600（大于 d_model），秩会是多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> d 个硬币加起来，标准差根号 d，所以除根号 d；
> softmax 太尖梯度没，太平学不出；
> 注意力就是核回归，温度就是核宽；
> 每个头秩不过 d_head，多头是拆开不是加；
> 因果 mask 先填负无穷，再 softmax。